# Séparation de sources audio par MNMF convolutif : implémentation de l’algorithme de Ozerov & Févotte (2010)


### Chargement d’un morceau depuis MUSDB18

Dans ce premier bloc, je commence par charger un morceau de la base de données **MUSDB18**, une référence en séparation de sources musicales. J’utilise pour cela la bibliothèque `musdb`, qui me permet d’accéder facilement aux différentes pistes ("stems") d’un morceau.

Je crée une instance de la base avec `musdb.DB`, en pointant vers le dossier contenant les fichiers `.stem.mp4`.

Je sélectionne le **premier morceau** de la base via `track = db.tracks[0]`.

Chaque piste (`drums`, `bass`, `other`, `vocals`) ainsi que le **mix complet** sont ensuite chargés sous forme de tableaux NumPy de forme `(N, 2)` correspondant aux canaux gauche et droit (stéréo).

Ce chargement me permet d’avoir un contrôle direct sur les sources individuelles du morceau, ce qui est fondamental pour la simulation de mélanges convolutifs ultérieurs et pour l’évaluation des performances de séparation.


In [ ]:
import musdb

# Chargement de la base MUSDB18 (fichiers locaux)
db = musdb.DB(download=False, is_wav=False, root='MUSDB18')

# Sélection du premier morceau
track = db.tracks[0]

# Extraction des différentes pistes (stéréo)
mix     = track.audio
drums   = track.targets['drums'].audio
bass    = track.targets['bass'].audio
other   = track.targets['other'].audio
vocals  = track.targets['vocals'].audio


### 1. Extraire les stems MUSDB à la volée (Python only)

### Extraction de segments mono

Je prépare les signaux mono nécessaires à la simulation. À partir du premier morceau MUSDB18, j’extrais 15 secondes de voix et d’accompagnement (à partir de la 10e seconde), que je moyenne sur les deux canaux pour obtenir des signaux mono.

Les fichiers `vocals_mono.wav` et `guitar_mono.wav` sont ensuite sauvegardés dans un dossier de travail (`work/`). Ces fichiers serviront à simuler une prise multicanale avec convolution.


In [ ]:
import musdb, soundfile as sf, pathlib

# Chargement du morceau
root = pathlib.Path("MUSDB18")
work = pathlib.Path("work"); work.mkdir(exist_ok=True)
db = musdb.DB(download=False, is_wav=False, root=root)
track = db.tracks[0]

# Paramètres d’extraction
sr = track.rate
dur = 15
beg = 10

def save_stem(name, audio):
    seg = audio[int(beg*sr):int((beg+dur)*sr)]
    sf.write(work/f"{name}.wav", seg, sr)

# Création des stems mono (moyenne des canaux)
save_stem("vocals_mono", track.targets["vocals"].audio.mean(axis=1))
save_stem("guitar_mono", track.targets["other"].audio.mean(axis=1))

# Lecture audio dans le notebook
import IPython.display as ipd

print("Voix (mono) :")
display(ipd.Audio(work / "vocals_mono.wav"))

print("Guitare (mono) :")
display(ipd.Audio(work / "guitar_mono.wav"))


### 2. Récupérer et préparer deux RIRs (AIR)

Nous utilisons deux filtres différents pour 

### Chargement des réponses impulsionnelles (RIR)

Je charge quatre réponses impulsionnelles issues de la base AIR (Aachen Impulse Response). Chaque fichier `.mat` contient une RIR mono ainsi que sa fréquence d’échantillonnage.

La fonction `load_air_channel` lit la structure `h_air` et la fréquence `fs`, que je réutilise pour simuler la propagation des sources vers deux micros virtuels placés à différents azimuts.


In [ ]:
from scipy.io import loadmat, wavfile
import numpy as np

# Chargement d’un RIR depuis un fichier .mat
def load_air_channel(path):
    mat = loadmat(path, squeeze_me=True)
    h   = mat['h_air']
    fs  = int(mat['air_info']['fs'])
    return h, fs

# Sélection de 4 réponses impulsionnelles depuis la base AIR
rir1, fs = load_air_channel("AIR_1_4/air_binaural_aula_carolina_0_1_1_90_3.mat")
rir2, _  = load_air_channel("AIR_1_4/air_binaural_aula_carolina_0_1_3_0_3.mat")
rir3, _  = load_air_channel("AIR_1_4/air_binaural_aula_carolina_0_1_3_180_3.mat")
rir4, _  = load_air_channel("AIR_1_4/air_binaural_aula_carolina_0_1_3_135_3.mat")


### Lecture audio des réponses impulsionnelles

J’écoute une des réponses impulsionnelles chargées à l’étape précédente pour en évaluer le contenu. Cela me permet de vérifier que le signal est bien chargé et exploitable.

J’applique ensuite un simple facteur de gain pour amplifier le signal (`* 1000`), ce qui permet d’entendre plus clairement la réverbération, tout en faisant attention à ne pas saturer l’amplitude.


In [ ]:
import IPython.display as ipd
ipd.Audio(rir1, rate=fs)
rir_boost = rir1 * 1000          # amplifier 
ipd.Audio(rir_boost, rate=fs)


### Rééchantillonnage des RIR

Les réponses impulsionnelles téléchargées sont parfois à 48 kHz, tandis que mes signaux audio sont à 44,1 kHz. Pour éviter tout décalage temporel ou artefact spectral, je vérifie la fréquence de chaque RIR et je la rééchantillonne si nécessaire à l’aide de `librosa.resample`.

Je traite ainsi les quatre RIR utilisées pour simuler la propagation des deux sources vers deux microphones.


In [ ]:
import librosa

sr_target = sr  # fréquence cible : même que les stems

# Si besoin, on rééchantillonne les RIR à 44.1 kHz
def resample_if_needed(rir_vec, fs_rir, sr_target):
    if fs_rir != sr_target:
        rir_vec = librosa.resample(rir_vec, orig_sr=fs_rir, target_sr=sr_target)
    return rir_vec

# Rééchantillonnage des 4 RIR utilisés
rir_v_mic1 = resample_if_needed(rir1, fs, sr_target)
rir_g_mic1 = resample_if_needed(rir2, fs, sr_target)
rir_v_mic2 = resample_if_needed(rir3, fs, sr_target)
rir_g_mic2 = resample_if_needed(rir4, fs, sr_target)


### Simulation des signaux de microphones par convolution

Je commence par charger les deux stems mono (voix et guitare) extraits précédemment. Avant la convolution, je normalise chaque RIR pour que toutes aient un gain RMS comparable, ce qui évite de biaiser le mélange.

Je simule ensuite deux microphones. Chaque micro capte à la fois la voix et la guitare, chacune convoluée avec sa propre réponse impulsionnelle. Cela revient à modéliser un mélange convolutif, avec deux sources et deux micros.

Je sauvegarde enfin les deux signaux obtenus (`mic1.wav` et `mic2.wav`) pour les utiliser dans la suite du traitement.


In [ ]:
import numpy as np, soundfile as sf
from scipy.signal import fftconvolve

# Lecture des stems mono
sr = 44100                           # taux d'échantillonnage cible
voc_mono, _ = sf.read(work/"vocals_mono.wav")
git_mono, _ = sf.read(work/"guitar_mono.wav")

# Normalisation des RIR pour éviter les différences de gain
def norm_rir(r):
    return r / np.sqrt(np.sum(r**2) + 1e-9)

rir_v_mic1 = norm_rir(rir1)
rir_g_mic1 = norm_rir(rir2)
rir_v_mic2 = norm_rir(rir3)
rir_g_mic2 = norm_rir(rir4)

# Convolution FFT des sources avec les RIR
def conv(src, rir):
    return fftconvolve(src, rir)[:len(src)]

mic1 = conv(voc_mono, rir_v_mic1) + conv(git_mono, rir_g_mic1)
mic2 = conv(voc_mono, rir_v_mic2) + conv(git_mono, rir_g_mic2)

# Enregistrement des signaux simulés
mix = np.vstack([mic1, mic2])
sf.write(work/"mic1.wav", mic1, sr)
sf.write(work/"mic2.wav", mic2, sr)

print("→ mic1.wav et mic2.wav créés dans", work.resolve())


### Passage dans le domaine temps–fréquence (STFT)

Je charge les deux signaux de microphone simulés, puis j’applique une transformée de Fourier à court terme (STFT) avec `librosa`, en utilisant une fenêtre de 256 échantillons sans recouvrement.

Cela me permet d’obtenir les spectrogrammes complexes des deux signaux, que je regroupe dans un tenseur `X` de forme `(F, T, I)` :
- `F` : nombre de fréquences,
- `T` : nombre de trames temporelles,
- `I = 2` : nombre de canaux (micros).

Je calcule enfin la puissance spectrale observée `Px`, qui sera utilisée dans l’estimation statistique des sources.


In [ ]:
import numpy as np, soundfile as sf, librosa
from scipy.signal import stft, istft

# Lecture des deux signaux de micro
sig, sr = sf.read("work/mic1.wav")
mic1 = sig
mic2 = sf.read("work/mic2.wav")[0]

# STFT de chaque micro
X1 = librosa.stft(mic1, n_fft=256, hop_length=256, window='hann')
X2 = librosa.stft(mic2, n_fft=256, hop_length=256, window='hann')

# Empilement des STFT et permutation des axes pour (F, T, I)
X = np.stack([X1, X2], axis=0)
X = np.moveaxis(X, 0, -1)

# Dimensions et puissance spectrale
F, T, I = X.shape
J = 2
Px = np.abs(X) ** 2


### Définition des groupes de composantes NMF

Je définis ici la structure des sources latentes. Le tableau `K_set` indique quelles composantes NMF sont associées à chaque source.

Par exemple :
- la source 0 (voix) utilisera les composantes 0 et 1 ;
- la source 1 (guitare) utilisera les composantes 2 et 3.

Cela permet de lier chaque source à un sous-ensemble des composantes du modèle NMF. Je déduis ensuite `K`, le nombre total de composantes utilisées dans le modèle.


In [ ]:
# Regroupement des composantes K en deux groupes (une par source)
K_set = np.array([[0,1],[2,3]])
K = K_set.size  # nombre total de composantes NMF


### Initialisation des paramètres du modèle MNMF

J’initialise les paramètres du modèle multicanal convolutif :

- `W` (F × K) : les motifs spectraux non négatifs, initialisés aléatoirement.
- `H` (K × T) : les activations temporelles, également aléatoires.
- `A` (F × I × J) : les filtres de mélange complexes, représentant la propagation des sources vers les micros pour chaque fréquence. Chaque vecteur colonne est normalisé en norme 2 pour éviter une échelle arbitraire.

Je fixe aussi une estimation initiale du bruit `sigma²`, proportionnelle à la moyenne de la puissance spectrale observée. Enfin, `Sigma_b` est la matrice de covariance du bruit, supposée diagonale et identique pour toutes les fréquences.


In [ ]:
# Initialisation des paramètres
rng = np.random.default_rng(0)
W = rng.random((F, K)) + 1e-2       # matrice des motifs
H = rng.random((K, T)) + 1e-2       # activations
A = rng.random((F, I, J)) + 1j * rng.random((F, I, J))  # mélange spatial
A /= np.linalg.norm(A, axis=1, keepdims=True)          # normalisation
sigma2 = 1e-4 * Px.mean()
Sigma_b = sigma2 * np.eye(I)


In [ ]:
# --------------------------------------------------
# Vérification des dimensions
# --------------------------------------------------
print("--------- dimensions internes ---------------")
print(f"C (nb micros)   : {I}")          # généralement 2
print(f"F (bins FFT)    : {F}")          # n_fft//2 + 1
print(f"T (trames)      : {T}")          # dépend de la durée / hop
print(f"K (composantes) : {K}")          # ton choix (ex. 10) 
print("---------------------------------------------")
print("X   shape :", X.shape,  "  (F, T, I)")
print("W   shape :", W.shape,  "  (F, K)")
print("H   shape :", H.shape,  "  (K, T)")
print("A   shape :", A.shape,  "  (F, I, J)")
print("---------------------------------------------")
print(f"sigma² scalaire : {sigma2:.3e}")
print("---------------------------------------------")

print(f"Nombre de sources: {J}")


### Construction de la version augmentée des filtres `A`

Je construis ici la matrice `A_aug`, une version étendue des filtres spatiaux `A`, utilisée pour les calculs dans l'étape E de l’algorithme EM.

Chaque composante latente `k` est associée au filtre de sa source correspondante `j`. Ainsi, pour chaque fréquence, `A_aug[:, :, k]` est une copie de `A[:, :, j]` si `k ∈ K_set[j]`.

Cette réorganisation permet de traiter toutes les composantes individuellement tout en conservant leur liaison avec les sources physiques.


In [ ]:
# Construction de la version augmentée de A
def A_augmentation(A, K_set):
    A_aug = np.zeros((A.shape[0], A.shape[1], K), dtype=complex)
    for j in range(len(K_set)):
        for k in K_set[j]:
            A_aug[:, :, k] = A[:, :, j]
    return A_aug

A_aug = A_augmentation(A, K_set)
print(A_aug.shape)


### Étape E de l’algorithme EM (Expectation Step)

Dans cette partie, j’implémente l’étape E de l’algorithme EM. Pour chaque fréquence `f`, je boucle sur chaque trame temporelle `t` afin de calculer les statistiques d'espérance nécessaires.

#### Calculs principaux :

- `Sigma_c` : matrice de covariance des composantes NMF `k`, basée sur les produits `W[f,k]*H[k,t]`.
- `Sigma_s` : matrice de covariance des sources `j`, en agrégeant les composantes associées via `K_set`.
- `Sigma_x` : matrice de covariance totale du signal observé, obtenue via la propagation des sources à travers les filtres `A[f]`, plus le bruit `Sigma_b`.

À partir de ces matrices, je calcule les estimateurs linéaires :

- `Gc` : gain pour estimer les composantes latentes individuelles (`u`).
- `Gs` : gain pour estimer les sources latentes (`s`).

Ces estimateurs me permettent de calculer les moments nécessaires :

- `Rxx`, `Rxs`, `Rss` : estimations des corrélations croisées utiles à l'étape M.
- `u[k,f,t]` : énergie moyenne de chaque composante latente, indispensable pour mettre à jour les matrices `W` et `H`.

J’ajoute également plusieurs étapes de normalisation de `A` (par norme et phase), et je mets à jour la matrice de covariance du bruit `Sigma_b` par la diagonale résiduelle de `Rxx`.

Cette étape constitue le cœur du modèle probabiliste, où j’infère les variables latentes à partir du mélange observé.


In [ ]:
# Paramètres de forme
F, T, I = X.shape
eps = 1e-9

# Initialisation de la variable auxiliaire u
A_aug = A_augmentation(A, K_set)
u = np.zeros((K, F, T))

for f in range(F):
    # Matrices pour l'accumulation
    Rxx = np.zeros((I, I), dtype=complex)
    Rxs = np.zeros((I, J), dtype=complex)
    Rss = np.zeros((J, J), dtype=complex)
    
    for t in range(T):
        # Covariances des composantes et des sources
        Sigma_c = np.diag([W[f, k] * H[k, t] for k in range(K)])
        Sigma_s = np.diag([np.sum([W[f, k] * H[k, t] for k in Kj]) for Kj in K_set])
        Sigma_x = A[f] @ Sigma_s @ A[f].conj().T + Sigma_b

        # Estimateurs a posteriori (Eq. 15–16)
        Gc = Sigma_c @ A_aug[f].conj().T @ np.linalg.inv(Sigma_x)
        c = Gc @ X[f, t]

        Gs = Sigma_s @ A[f].conj().T @ np.linalg.inv(Sigma_x)
        s = Gs @ X[f, t]

        # Mise à jour des moments (Eqs. 17–18)
        Rxx += X[f, t] @ X[f, t].conj().T
        Rxs += X[f, t] @ s.conj().T
        Rss += s @ s.conj().T + Sigma_s - Gs @ A[f] @ Sigma_s

        # Mise à jour de u (utilisé pour W,H)
        Mat_u = c @ c.conj().T + Sigma_c - Gc @ A_aug[f] @ Sigma_c
        u[:, f, t] = np.diag(Mat_u)

    # Normalisation
    Rxx /= T
    Rxs /= T
    Rss /= T

    # M-step : mise à jour spatiale (A) (Eq. 26)
    A[f] = Rxs @ np.linalg.inv(Rss)

    # Normalisations
    for j in range(J):
        norm = np.sum(np.sqrt(np.abs(A[f, :, j])**2))
        A[f, :, j] /= norm
        W[f, K_set[j]] *= norm

        ref = A[f, 0, j]
        A[f, :, j] /= ref / np.abs(ref)

    for k in range(K):
        scale = np.sum(W[:, k])
        W[:, k] /= scale
        H[k, :] *= scale

    for j in range(J):
        A[f, :, j] /= np.sum(np.abs(A[f, :, j])**2)

    # Mise à jour du bruit résiduel
    Sigma_b = np.diag(np.diag(Rxx - A[f] @ Rxs.conj().T - Rxs @ A[f].conj().T + A[f] @ Rss @ A[f].conj().T))


### Maximisation de W et H

Dans cette dernière partie de la boucle EM, j’effectue la mise à jour explicite des matrices `W` et `H`, selon les équations données dans la section A.4 du papier (équation 26).


In [ ]:
# Maximisation des paramètres spectraux W et H (Eq. 26)

for f in range(F):  
    for k in range(K):
        # Mise à jour de W[f, k] avec la moyenne des rapports u/H
        W[f, k] = np.mean(u[k, f, :] / H[k, :])

for k in range(K):
    for t in range(T):
        # Mise à jour de H[k, t] avec la moyenne des rapports u/W
        H[k, t] = np.mean(u[k, :, t] / W[:, k])


### Reconstruction des spectrogrammes sources

Je calcule le tenseur `V` de forme `(F, K, T)` représentant les composantes spectro-temporelles estimées par la NMF.

Ensuite, je regroupe les composantes par source à l’aide de `K_set` pour obtenir une liste `V_j` contenant les spectrogrammes estimés `(F, T)` de chaque source. Cela me permet d’isoler l’énergie associée à chaque source dans le domaine temps-fréquence.


In [ ]:
# Recalcule des spectrogrammes de puissance V pour chaque composante k
V = np.array([W[:, k][:, None] * H[k][None, :] for k in range(K)])  # (K, F, T)

# On remet les dimensions dans l'ordre (F, K, T)
V = np.transpose(V, (1, 0, 2))  # (F, K, T)

# Agrégation des composantes par source j, pour obtenir un V_j par source
V_j = [np.sum(V[:, kj, :], axis=1) for kj in K_set]  # liste de (F, T)


### Estimation des matrices de covariance Rj

Pour chaque source `j`, je construis la matrice de covariance spectro-temporelle `Rj`, qui représente la contribution de cette source au signal observé.

Je commence par extraire la matrice de mélange spatiale `A[:, :, j]` et le spectrogramme estimé `V_j[j]`.  
Pour chaque bin fréquentiel `f` et trame temporelle `t`, je calcule la matrice de covariance spatiale comme :
$
R_j(f, t) = v_j(f, t) \cdot a_j(f) a_j(f)^H
$
Ce tenseur `(F, I, I, T)` encapsule les propriétés spatiales et spectrales de la source estimée.


In [ ]:
# Construction des matrices de covariance spatiale pour chaque source
Rj = []

for j in range(J):
    a = A[:, :, j]    # matrice de mélange pour la source j, de taille (F, I)
    v = V_j[j]        # spectrogramme de puissance estimé pour la source j, (F, T)
    
    Rjt = np.zeros((F, I, I, T), dtype=complex)  # initialisation de Rj(f,t)
    
    for f in range(F):
        af = a[f][:, None]  # vecteur colonne (I, 1)
        for t in range(T):
            # On applique la formule R_j[f,t] = v[f,t] * a[f] @ a[f].H
            Rjt[f, :, :, t] = v[f, t] * (af @ af.conj().T)
    
    # On stocke la séquence de matrices pour la source j
    Rj.append(Rjt)


### Estimation des sources séparées

Je commence par calculer la covariance totale du mélange `Rx` en sommant les contributions de chaque source `Rj`, puis j’ajoute la covariance du bruit `Sigma_b`.

Pour chaque source `j`, j’applique un masque Wiener multicanal, basé sur la formule :
\[
M_{j}(f, t) = R_j(f, t) \cdot R_x(f, t)^{-1}
\]
Ce masque est appliqué à l’observation spectrale `X[f, t]` afin d’estimer la contribution spectrale de chaque source. Le résultat est stocké dans la liste `S_hat`, sous forme de tenseurs complexes `(F, T, I)`, un par source.


In [ ]:
# Reconstruction des sources par filtrage de Wiener multicanal

# On commence par sommer les matrices de covariance Rj pour obtenir Rx,
# la covariance totale du mélange à chaque bin de fréquence et trame de temps.
Rx = np.sum(np.stack(Rj), axis=0)  # Rx : (F, I, I, T)

# On ajoute le bruit stationnaire Sigma_b sur la diagonale
for f in range(F):
    for t in range(T):
        Rx[f, :, :, t] += Sigma_b

# Reconstruction de chaque source estimée
S_hat = []
for j in range(J):
    Sj = np.zeros_like(X, dtype=complex)  # Initialisation (F, T, I)
    for f in range(F):
        for t in range(T):
            # Calcul du masque Wiener multicanal pour la source j
            M = Rj[j][f, :, :, t] @ np.linalg.inv(Rx[f, :, :, t])
            # Application de ce filtre au mélange X pour obtenir l'estimation de la source j
            Sj[f, t, :] = M @ X[f, t, :]
    S_hat.append(Sj)


### Reconstruction temporelle des sources estimées

Pour chaque source estimée `S_hat[j]`, je réorganise les dimensions pour obtenir un tableau de forme `(I, F, T)` : un spectrogramme par micro.

J’applique ensuite l’inverse de la STFT (via `librosa.istft`) à chaque canal pour revenir dans le domaine temporel. Les signaux reconstruits sont sauvegardés dans le dossier `work/` sous le nom `source{j}_mic{i}.wav`.

Cela me permet d’écouter les signaux séparés pour chaque source et chaque micro.


In [ ]:
import librosa

# Reconstruction temporelle des sources séparées
for j, Sj in enumerate(S_hat):
    Sj = np.moveaxis(Sj, -1, 0)  # passage de (F, T, I) à (I, F, T)
    for i in range(I):
        # ISTFT canal par canal
        y = librosa.istft(Sj[i], hop_length=256, window='hann')
        sf.write(f"work/source{j}_mic{i}.wav", y, sr)


### Conclusion

Dans ce TP, j’ai implémenté une version simplifiée de l’algorithme EM présenté dans l’article *Multichannel NMF in Convolutive Mixtures for Audio Source Separation* de Ozerov et Févotte. J’ai d’abord simulé un mélange stéréo réverbéré à partir de pistes issues de la base MUSDB18 et de réponses impulsionnelles de la base AIR. J’ai ensuite codé les différentes étapes du modèle probabiliste : initialisation, boucle EM (estimation des espérances, mise à jour des paramètres), reconstruction spectrale puis temporelle.

Cependant, les résultats obtenus sont décevants : les signaux reconstruits ne montrent pas de séparation claire des sources. Plusieurs hypothèses peuvent expliquer cet échec :

- Les valeurs de la variable auxiliaire `u`, qui sert à mettre à jour les matrices NMF `W` et `H`, sont potentiellement **complexes**, ce qui contredit leur nature supposée réelle et positive dans le modèle original.
- Le **coût de calcul** d’une seule itération de l’algorithme EM est élevé, rendant difficile l’expérimentation sur plusieurs itérations. Des optimisations matricielles ou des implémentations plus efficaces (GPU, PyTorch) seraient nécessaires pour accélérer la convergence.
- Des **erreurs d’initialisation**, de dimensions ou de stabilité numérique ont pu se glisser dans le pipeline, notamment lors de la construction des matrices de covariance ou de l’application des contraintes de normalisation.
- Enfin, la **reconstruction temporelle** est sensible à de petites erreurs dans les masques spectraux ou dans la manipulation des phases, ce qui peut dégrader fortement la qualité audio finale.

Malgré ces limites, ce travail m’a permis de mieux comprendre le fonctionnement interne des modèles MNMF convolutifs et les défis pratiques de leur mise en œuvre. Il constitue une base que je pourrais améliorer ultérieurement, en corrigeant les erreurs identifiées et en ajoutant des tests plus systématiques à chaque étape.
